In [5]:
pip install sqlalchemy pymysql

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import pandas as pd
import numpy as np

In [3]:
print("=== BLOCK 1: PROCESSING JOBS BY INDUSTRY ===")

# Processing Industry Employment Data
jobs_file = pd.read_csv("Jobs_by_industry_in_Bristol_by_Ward.csv")

# Group by ward to get the most recent structural workforce snapshot (2023)
jobs_2023 = jobs_file[jobs_file['YEAR'] == 2023].copy()

# Define the three distinct socioeconomic sectors
knowledge_sectors = [
    'INFORMATION_COMMUNICATION', 'FINANCIAL_INSURANCE', 'PROPERTY', 
    'PROFESSIONAL_SCI_TECH', 'BUSINESS_ADMIN_SUPPORT'
]

traditional_sectors = [
    'MANUFACTURING', 'CONSTRUCTION', 'MOTOR_TRADES', 'TRANSPORT_STORAGE'
]

population_serving_sectors = [
    'RETAIL', 'WHOLESALE', 'ACCOMMODATION_FOOD', 'EDUCATION', 'HEALTH', 
    'ARTS_ENTS_RECREATION_OTHER', 'PUBLIC_ADMIN_DEFENCE', 'MINING_UTILITIES', 
    'AG_FORESTRY_FISHING'
]

# Feature Engineering: Calculate sums across sectors
jobs_2023['total_estimated_jobs'] = jobs_2023.iloc[:, 4:22].sum(axis=1)
jobs_2023['knowledge_jobs_sum'] = jobs_2023[knowledge_sectors].sum(axis=1)
jobs_2023['traditional_jobs_sum'] = jobs_2023[traditional_sectors].sum(axis=1)
jobs_2023['pop_serving_jobs_sum'] = jobs_2023[population_serving_sectors].sum(axis=1)

# Feature Engineering: Calculate percentages relative to the total workforce
jobs_2023['pc_knowledge_jobs'] = (jobs_2023['knowledge_jobs_sum'] / jobs_2023['total_estimated_jobs']) * 100
jobs_2023['pc_traditional_jobs'] = (jobs_2023['traditional_jobs_sum'] / jobs_2023['total_estimated_jobs']) * 100
jobs_2023['pc_pop_serving_jobs'] = (jobs_2023['pop_serving_jobs_sum'] / jobs_2023['total_estimated_jobs']) * 100

# Select clean output columns for our database staging file
clean_jobs = jobs_2023[[
    'WARD_CODE', 'WARD_NAME', 'total_estimated_jobs', 
    'pc_knowledge_jobs', 'pc_traditional_jobs', 'pc_pop_serving_jobs'
]].copy()

# Convert column names to lowercase for clean SQL integration
clean_jobs.columns = clean_jobs.columns.str.lower()
clean_jobs.to_csv("staged_ward_jobs.csv", index=False)

print("SUCCESS: 'staged_ward_jobs.csv' updated with the full 3-way framework")
print(clean_jobs.head(5))
print("\n=== BLOCK 1 COMPLETE ===")

=== BLOCK 1: PROCESSING JOBS BY INDUSTRY ===
SUCCESS: 'staged_ward_jobs.csv' updated with the full 3-way framework
     ward_code                      ward_name  total_estimated_jobs  \
272  E05010885                         Ashley                 10115   
273  E05010886  Avonmouth and Lawrence Weston                 19910   
274  E05010887                     Bedminster                 10190   
275  E05010888     Bishopston and Ashley Down                  3260   
276  E05010889                   Bishopsworth                  7355   

     pc_knowledge_jobs  pc_traditional_jobs  pc_pop_serving_jobs  
272          48.690064             6.327237            44.982699  
273          12.506278            46.459066            41.034656  
274          15.112856            29.440628            55.446516  
275          16.411043             7.668712            75.920245  
276          63.698165            14.072060            22.229776  

=== BLOCK 1 COMPLETE ===


In [4]:
print("=== BLOCK 2: PROCESSING SCHOOLS INFRASTRUCTURE ===")

# Define the file directory locally
file_directory = "./" 

# Read the raw schools directory using safety encoding
schools_filename = "Schools in Bristol (All).csv"
full_schools_path = os.path.join(file_directory, schools_filename)

if not os.path.exists(full_schools_path):
    print(f" ERROR: Could not find the file at: {os.path.abspath(full_schools_path)}")
else:
    # Read the file bypassing byte errors
    schools_raw = pd.read_csv(full_schools_path, encoding="latin1")
    
    # Filter for Open schools specifically under Bristol local authority management
    bristol_schools = schools_raw[
        (schools_raw['EstablishmentStatus (name)'] == 'Open') & 
        (schools_raw['LA (name)'].str.contains('Bristol', na=False, case=False))
    ].copy()
    
    # Group by the official Administrative Ward names to calculate absolute school density
    schools_by_ward = bristol_schools.groupby('AdministrativeWard (name)').size().reset_index(name='total_schools')
    
    # Standardize column titles to lowercase for smooth SQL ingestion later
    schools_by_ward.columns = ['ward_name', 'total_schools']
    
    # Save the second staging CSV
    schools_output_path = os.path.join(file_directory, "staged_ward_schools.csv")
    schools_by_ward.to_csv(schools_output_path, index=False)
    
    print(" SUCCESS: 'staged_ward_schools.csv' generated cleanly with real ONS wards!")
    print(schools_by_ward.head(5))

    # Getting a unique code-to-name bridge from the clean_jobs DataFrame 
    ward_bridge = clean_jobs[['ward_code', 'ward_name']].drop_duplicates()
    
    # Merge the ward codes into the schools data framework
    final_schools = pd.merge(ward_bridge, schools_by_ward, on='ward_name', how='left')
    
    # Fill any missing values with 0 (in case a ward has no open schools listed)
    final_schools['total_schools'] = final_schools['total_schools'].fillna(0).astype(int)
    
    # Save the structurally complete staging CSV
    schools_output_path = os.path.join(file_directory, "staged_ward_schools.csv")
    final_schools.to_csv(schools_output_path, index=False)
    
    print("SUCCESS: 'staged_ward_schools.csv' generated with official ONS codes!")
    print(final_schools.head(5))
    print("\n=== BLOCK 2 COMPLETE ===")

=== BLOCK 2: PROCESSING SCHOOLS INFRASTRUCTURE ===
 SUCCESS: 'staged_ward_schools.csv' generated cleanly with real ONS wards!
                       ward_name  total_schools
0                         Ashley             10
1  Avonmouth and Lawrence Weston             14
2                     Bedminster              5
3     Bishopston and Ashley Down              2
4                   Bishopsworth              5
SUCCESS: 'staged_ward_schools.csv' generated with official ONS codes!
   ward_code                      ward_name  total_schools
0  E05010885                         Ashley             10
1  E05010886  Avonmouth and Lawrence Weston             14
2  E05010887                     Bedminster              5
3  E05010888     Bishopston and Ashley Down              2
4  E05010889                   Bishopsworth              5

=== BLOCK 2 COMPLETE ===


In [5]:
from sqlalchemy import create_engine, types

print("=== BLOCK 3: AUTOMATED DATABASE MIGRATION ===")

# Database Connection Credentials 
db_user = "root"          # MySQL username
db_password = "Bristol_2026"  # MySQL password
db_host = "localhost"     
db_port = "3306"         
db_name = "bristol_gentrification"  # The name of database schema

# Create the SQLAlchemy engine string
engine_url = f"mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"

try:
    engine = create_engine(engine_url)
    print("Connection to MySQL database established successfully!")
    
    # Migrate Block 1 Data: Jobs by Industry
    print("Uploading staging_ward_jobs...")
    clean_jobs.to_sql(
        name="staging_ward_jobs",
        con=engine,
        if_exists="replace", 
        index=False,         
        dtype={
            'ward_code': types.VARCHAR(20),
            'ward_name': types.VARCHAR(100),
            'total_estimated_jobs': types.INTEGER(),
            'pc_knowledge_jobs': types.Numeric(5, 2),
            'pc_traditional_jobs': types.Numeric(5, 2),
            'pc_pop_serving_jobs': types.Numeric(5, 2)
        }
    )
    print("Table 'staging_ward_jobs' successfully updated in MySQL.")

    # Migrate Block 2 Data: Schools Density Framework
    print("Uploading staging_ward_schools...")
    final_schools.to_sql(
        name="staging_ward_schools",
        con=engine,
        if_exists="replace",
        index=False,
        dtype={
            'ward_code': types.VARCHAR(20),
            'ward_name': types.VARCHAR(100),
            'total_schools': types.INTEGER()
        }
    )
    print("Table 'staging_ward_schools' successfully updated in MySQL.")
    
    print("\nPIEPLINE STATUS: ALL SUPPLEMENTARY DATASETS MIGRATED TO SQL SUCESSFULLY!")

except Exception as e:
    print(f"DATABASE ERROR: Could not complete migration. Reason: {e}")

print("\n=== BLOCK 3 COMPLETE ===")

=== BLOCK 3: AUTOMATED DATABASE MIGRATION ===
Connection to MySQL database established successfully!
Uploading staging_ward_jobs...
Table 'staging_ward_jobs' successfully updated in MySQL.
Uploading staging_ward_schools...
Table 'staging_ward_schools' successfully updated in MySQL.

PIEPLINE STATUS: ALL SUPPLEMENTARY DATASETS MIGRATED TO SQL SUCESSFULLY!

=== BLOCK 3 COMPLETE ===
